# 📋 Review Data Collector — Google & Yelp
**For marketing agency use | Exports a ready-to-upload spreadsheet**

---

## Before You Start: 3-Step Setup Guide

This notebook pulls reviews for your client and their competitors from both Google and Yelp, then exports everything into one clean spreadsheet. You only need **one API key** to make it work.

### Step 1 — Create a free Apify account
Go to **[apify.com](https://apify.com)** and sign up for a free account. The free plan gives you enough credits to run this tool regularly for a small number of businesses.

### Step 2 — Copy your Apify API token
1. Log in to Apify
2. Click your profile icon in the top-right corner
3. Go to **Settings → Integrations**
4. Under **API tokens**, copy the token shown (it starts with `apify_api_...`)
5. You'll paste this into the Config cell below

### Step 3 — Fill in the Config cell
Scroll down to **Cell 3 (⚙️ Configuration)**. Fill in:
- Your Apify API token
- Your **client's** business name, Google Maps URL, and Yelp URL
- Up to 5 **competitors** with the same info

Then run all cells from top to bottom using **Runtime → Run all**.

---

### How to find the right URLs

**Google Maps URL:**
1. Go to [maps.google.com](https://maps.google.com)
2. Search for the business by name
3. Click on the business listing
4. Copy the full URL from your browser's address bar
   - It will look like: `https://www.google.com/maps/place/Business+Name/@...`

**Yelp URL:**
1. Go to [yelp.com](https://yelp.com)
2. Search for the business
3. Click on the business listing
4. Copy the full URL from your browser's address bar
   - It will look like: `https://www.yelp.com/biz/business-name-city`

> **Tip:** If a competitor isn't on Yelp (or Google), just leave that URL blank — the notebook will skip it automatically.

---
## Cell 1 — Install Required Tools

This cell installs the behind-the-scenes tools that Python needs to run this notebook. You don't need to change anything here — just run it and wait for it to finish. You'll see a ✅ message when it's ready.

In [ ]:
# Install required libraries
import subprocess, sys

print("Installing required tools... this takes about 15 seconds.")

packages = ["requests", "pandas", "openpyxl"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import requests
import pandas as pd
import time
import json
from datetime import datetime

print("✅ All tools installed and ready!")

---
## Cell 2 — ⚙️ Configuration (Fill This In!)

**This is the only cell you need to edit.** Fill in your Apify API token, your client's info, and any competitors you want to include.

- **Leave a URL blank** (empty quotes `""`) if a business doesn't have a Google or Yelp listing.
- **Leave a competitor block blank** if you have fewer than 5 competitors — those will be skipped automatically.
- Business names can be anything you want — they'll appear in the spreadsheet exactly as you type them.

In [ ]:
# =============================================================================
#  ⚙️  CONFIGURATION — Fill in everything between the quotes below
# =============================================================================

# -------------------------------------------------------------------
# YOUR APIFY API TOKEN
# Find this at: apify.com → Settings → Integrations → API tokens
# It starts with "apify_api_..."
# -------------------------------------------------------------------
APIFY_API_TOKEN = "YOUR_APIFY_TOKEN_HERE"  # ← Paste your token here


# -------------------------------------------------------------------
# YOUR CLIENT
# -------------------------------------------------------------------
CLIENT = {
    "name":        "Client Business Name",        # ← Business name (you choose the label)
    "google_url":  "https://www.google.com/maps/place/...",  # ← Google Maps URL
    "yelp_url":    "https://www.yelp.com/biz/...",           # ← Yelp URL
}


# -------------------------------------------------------------------
# COMPETITORS (up to 5)
# Leave name as "" to skip that competitor slot entirely.
# Leave a URL as "" if that platform listing doesn't exist.
# -------------------------------------------------------------------
COMPETITORS = [
    {
        "name":        "Competitor 1 Name",
        "google_url":  "https://www.google.com/maps/place/...",
        "yelp_url":    "https://www.yelp.com/biz/...",
    },
    {
        "name":        "Competitor 2 Name",
        "google_url":  "",   # ← Leave blank if no Google listing
        "yelp_url":    "https://www.yelp.com/biz/...",
    },
    {
        "name":        "",   # ← Leave blank to skip this slot
        "google_url":  "",
        "yelp_url":    "",
    },
    {
        "name":        "",
        "google_url":  "",
        "yelp_url":    "",
    },
    {
        "name":        "",
        "google_url":  "",
        "yelp_url":    "",
    },
]


# =============================================================================
#  Stop editing here — nothing below needs to be changed
# =============================================================================

# Build the full list of businesses to scrape
ALL_BUSINESSES = []

if CLIENT.get("name"):
    ALL_BUSINESSES.append({**CLIENT, "label": "Client"})

for comp in COMPETITORS:
    if comp.get("name"):  # Skip empty competitor slots
        ALL_BUSINESSES.append({**comp, "label": "Competitor"})

# Validate token
if APIFY_API_TOKEN == "YOUR_APIFY_TOKEN_HERE" or not APIFY_API_TOKEN.strip():
    print("⚠️  Please paste your Apify API token into the APIFY_API_TOKEN field above and re-run this cell.")
else:
    print(f"✅ Configuration loaded!")
    print(f"   Client:      {CLIENT['name']}")
    active_competitors = [b for b in ALL_BUSINESSES if b['label'] == 'Competitor']
    if active_competitors:
        print(f"   Competitors: {', '.join(c['name'] for c in active_competitors)}")
    else:
        print(f"   Competitors: none configured")
    print(f"   Total businesses to scrape: {len(ALL_BUSINESSES)}")
    print()
    print("Ready to run! Proceed to the next cells.")

---
## Cell 3 — Pull Google Reviews

This cell connects to Apify and pulls Google reviews for every business you listed above. It goes through each business one at a time and waits for the data to come back before moving on.

Depending on how many businesses you have and how many reviews they have, **this cell may take 2–10 minutes to complete.** You'll see a live progress message for each business as it runs.

Just run this cell and wait — no action needed on your part.

In [ ]:
def run_apify_actor(actor_id, actor_input, token, poll_interval=10, timeout=600):
    """
    Starts an Apify actor run, waits for it to finish, and returns the dataset ID.
    Returns None if the run fails to start or complete.
    """
    url = f"https://api.apify.com/v2/acts/{actor_id}/runs?token={token}"
    try:
        resp = requests.post(url, json=actor_input, timeout=30)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"      ⚠️  Could not start actor: {e}")
        return None

    run_id = resp.json()["data"]["id"]
    default_dataset_id = resp.json()["data"]["defaultDatasetId"]

    # Poll until the run finishes
    elapsed = 0
    while elapsed < timeout:
        time.sleep(poll_interval)
        elapsed += poll_interval
        status_url = f"https://api.apify.com/v2/actor-runs/{run_id}?token={token}"
        try:
            status_resp = requests.get(status_url, timeout=30)
            status_resp.raise_for_status()
        except requests.RequestException as e:
            print(f"      ⚠️  Error checking run status: {e}")
            continue

        run_status = status_resp.json()["data"]["status"]
        if run_status == "SUCCEEDED":
            return default_dataset_id
        elif run_status in ("FAILED", "ABORTED", "TIMED-OUT"):
            print(f"      ⚠️  Actor run ended with status: {run_status}")
            return None

    print(f"      ⚠️  Timed out waiting for actor to finish after {timeout}s.")
    return None


def fetch_dataset_items(dataset_id, token, extra_params=""):
    """
    Downloads all items from an Apify dataset.
    Uses &view=reviews for Google Places to get flat, one-review-per-row format.
    """
    url = f"https://api.apify.com/v2/datasets/{dataset_id}/items?token={token}&format=json&clean=true{extra_params}"
    try:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        return resp.json()
    except requests.RequestException as e:
        print(f"      ⚠️  Could not fetch dataset items: {e}")
        return []


# ---------------------------------------------------------------------------
# Pull Google Reviews
# ---------------------------------------------------------------------------
print("="*60)
print("PULLING GOOGLE REVIEWS")
print("="*60)

all_google_reviews = []

businesses_with_google = [b for b in ALL_BUSINESSES if b.get("google_url", "").strip()]

if not businesses_with_google:
    print("No Google Maps URLs were provided — skipping Google reviews.")
else:
    for business in businesses_with_google:
        name   = business["name"]
        label  = business["label"]
        g_url  = business["google_url"].strip()

        print(f"\nPulling Google reviews for {name} ({label})...")

        actor_input = {
            "startUrls": [{"url": g_url}],
            "maxReviews": 500,       # cap per business — raise if you need more
            "reviewsSort": "newest",
            "language": "en",
            "scrapeReviewerName": True,
        }

        # NOTE: Apify actor IDs use ~ between owner and actor name in API calls
        dataset_id = run_apify_actor(
            actor_id="compass~crawler-google-places",
            actor_input=actor_input,
            token=APIFY_API_TOKEN,
        )

        if dataset_id is None:
            print(f"   ❌ Could not retrieve Google reviews for {name}. Skipping.")
            continue

        # Use &view=reviews to get one review per row (flat format)
        raw_items = fetch_dataset_items(dataset_id, APIFY_API_TOKEN, extra_params="&view=reviews")

        if not raw_items:
            print(f"   ⚠️  No Google reviews found for {name}.")
            continue

        reviews_found = 0
        for item in raw_items:
            # The reviews view returns review fields at the top level
            review_text = (
                item.get("text")
                or item.get("reviewText")
                or item.get("snippet")
                or ""
            )
            reviewer = (
                item.get("reviewerName")
                or item.get("name")
                or "Anonymous"
            )
            stars = item.get("stars") or item.get("rating") or item.get("starRating")
            pub_date = (
                item.get("publishedAtDate")
                or item.get("publishAt")
                or item.get("date")
                or ""
            )

            # Skip rows that don't look like actual reviews
            if stars is None and not review_text:
                continue

            all_google_reviews.append({
                "business_name":        name,
                "client_or_competitor": label,
                "platform":             "Google",
                "reviewer_name":        str(reviewer).strip(),
                "star_rating":          stars,
                "review_text":          str(review_text).strip(),
                "review_date":          str(pub_date)[:10] if pub_date else "",
            })
            reviews_found += 1

        print(f"   ✅ Done! Found {reviews_found} Google reviews for {name}.")

print(f"\n{'='*60}")
print(f"Google reviews complete. Total collected: {len(all_google_reviews)} reviews.")
print(f"{'='*60}")

---
## Cell 4 — Pull Yelp Reviews

This cell does the same thing as the previous one, but for Yelp. It goes through each business one at a time, waits for the data, and collects the reviews.

Again, this may take **2–10 minutes** depending on how many businesses you have. You'll see live progress updates.

Just run this cell and wait.

In [ ]:
# ---------------------------------------------------------------------------
# Pull Yelp Reviews
# ---------------------------------------------------------------------------
print("="*60)
print("PULLING YELP REVIEWS")
print("="*60)

all_yelp_reviews = []

businesses_with_yelp = [b for b in ALL_BUSINESSES if b.get("yelp_url", "").strip()]

if not businesses_with_yelp:
    print("No Yelp URLs were provided — skipping Yelp reviews.")
else:
    for business in businesses_with_yelp:
        name   = business["name"]
        label  = business["label"]
        y_url  = business["yelp_url"].strip()

        print(f"\nPulling Yelp reviews for {name} ({label})...")

        # web_wanderer/yelp-reviews-scraper uses "biz_urls" (not "startUrls")
        # include_personal_data must be True to receive reviewerName in output
        actor_input = {
            "biz_urls": [y_url],
            "include_personal_data": True,
        }

        dataset_id = run_apify_actor(
            actor_id="web_wanderer~yelp-reviews-scraper",
            actor_input=actor_input,
            token=APIFY_API_TOKEN,
        )

        if dataset_id is None:
            print(f"   ❌ Could not retrieve Yelp reviews for {name}. Skipping.")
            continue

        raw_items = fetch_dataset_items(dataset_id, APIFY_API_TOKEN)

        if not raw_items:
            print(f"   ⚠️  No Yelp reviews found for {name}.")
            continue

        reviews_found = 0
        for item in raw_items:
            review_text = (
                item.get("text")
                or item.get("reviewText")
                or item.get("comment")
                or ""
            )
            reviewer = (
                item.get("reviewerName")
                or item.get("userName")
                or item.get("author")
                or "Anonymous"
            )
            stars = (
                item.get("rating")
                or item.get("stars")
                or item.get("starRating")
                or item.get("ratingValue")
            )
            pub_date = (
                item.get("date")
                or item.get("publishedAt")
                or item.get("createdAt")
                or item.get("reviewDate")
                or ""
            )

            # Skip rows that don't look like actual reviews
            if stars is None and not review_text:
                continue

            # Normalize star rating to a number
            if isinstance(stars, str):
                try:
                    stars = float(stars)
                except ValueError:
                    stars = None

            all_yelp_reviews.append({
                "business_name":        name,
                "client_or_competitor": label,
                "platform":             "Yelp",
                "reviewer_name":        str(reviewer).strip(),
                "star_rating":          stars,
                "review_text":          str(review_text).strip(),
                "review_date":          str(pub_date)[:10] if pub_date else "",
            })
            reviews_found += 1

        print(f"   ✅ Done! Found {reviews_found} Yelp reviews for {name}.")

print(f"\n{'='*60}")
print(f"Yelp reviews complete. Total collected: {len(all_yelp_reviews)} reviews.")
print(f"{'='*60}")

---
## Cell 5 — Combine All Reviews

This cell takes all the Google reviews and all the Yelp reviews you just collected and merges them into one unified table.

It will show you a quick preview of the data so you can confirm everything looks right before exporting.

In [ ]:
# ---------------------------------------------------------------------------
# Combine Google + Yelp into one DataFrame
# ---------------------------------------------------------------------------
print("Combining Google and Yelp reviews...")

all_reviews = all_google_reviews + all_yelp_reviews

if not all_reviews:
    print("⚠️  No reviews were collected from either platform.")
    print("   Please check:")
    print("   1. Your Apify token is correct")
    print("   2. The Google Maps and Yelp URLs in the config cell are valid")
    print("   3. Your Apify account has available credits")
else:
    # Build the DataFrame with columns in the exact order needed
    COLUMNS = [
        "business_name",
        "client_or_competitor",
        "platform",
        "reviewer_name",
        "star_rating",
        "review_text",
        "review_date",
    ]

    df = pd.DataFrame(all_reviews, columns=COLUMNS)

    # Clean up star ratings — make sure they're numeric
    df["star_rating"] = pd.to_numeric(df["star_rating"], errors="coerce")

    # Remove completely empty rows
    df = df.dropna(subset=["review_text", "star_rating"], how="all").reset_index(drop=True)

    # Summary
    print(f"\n✅ Combined! Here's a summary of what was collected:")
    print(f"   Total reviews:    {len(df)}")
    print(f"   Google reviews:   {len(df[df['platform'] == 'Google'])}")
    print(f"   Yelp reviews:     {len(df[df['platform'] == 'Yelp'])}")
    print()

    # Breakdown by business
    print("   Breakdown by business:")
    summary = df.groupby(["business_name", "client_or_competitor", "platform"]).size().reset_index(name="count")
    for _, row in summary.iterrows():
        print(f"   • {row['business_name']} ({row['client_or_competitor']}) — {row['platform']}: {row['count']} reviews")

    print()
    print("Preview of first 5 rows:")
    display(df.head())

---
## Cell 6 — Export to Spreadsheet

This is the final step! This cell saves all the reviews into a single **.xlsx spreadsheet file** with today's date in the filename.

After it runs:
1. Look in the **Files panel** on the left sidebar of Colab (the folder icon)
2. Find the file named something like `reviews_export_2026-03-27.xlsx`
3. Right-click it and select **Download**
4. Upload the downloaded file to Google Sheets for sentiment analysis

> **Note:** Files in Colab are temporary and will disappear when your session ends, so make sure to download the file before closing this tab.

In [ ]:
# ---------------------------------------------------------------------------
# Export to .xlsx
# ---------------------------------------------------------------------------
try:
    # Make sure there's data to export
    if 'df' not in dir() or df is None or len(df) == 0:
        print("⚠️  No data to export. Please run the earlier cells first and make sure reviews were collected.")
    else:
        today = datetime.now().strftime("%Y-%m-%d")
        filename = f"reviews_export_{today}.xlsx"

        print(f"Saving {len(df)} reviews to {filename}...")

        with pd.ExcelWriter(filename, engine="openpyxl") as writer:
            df.to_excel(writer, index=False, sheet_name="Reviews")

            # Auto-size columns for readability
            worksheet = writer.sheets["Reviews"]
            col_widths = {
                "business_name":        30,
                "client_or_competitor": 20,
                "platform":             12,
                "reviewer_name":        25,
                "star_rating":          12,
                "review_text":          80,
                "review_date":          15,
            }
            for i, col_name in enumerate(df.columns, start=1):
                col_letter = worksheet.cell(row=1, column=i).column_letter
                worksheet.column_dimensions[col_letter].width = col_widths.get(col_name, 20)

        print(f"\n✅ Spreadsheet saved successfully!")
        print(f"   File name: {filename}")
        print(f"   Rows:      {len(df)}")
        print(f"   Columns:   {', '.join(df.columns.tolist())}")
        print()
        print("📥 To download:")
        print("   1. Click the folder icon in the left sidebar")
        print(f"   2. Find '{filename}'")
        print("   3. Right-click → Download")

        # Auto-trigger download if running in Colab
        try:
            from google.colab import files
            files.download(filename)
            print("\n   (Download should start automatically in your browser.)")
        except ImportError:
            pass  # Not running in Colab — file is saved locally

except Exception as e:
    print(f"❌ An error occurred while saving the file: {e}")
    print("   Try running this cell again. If the error persists, check that openpyxl is installed (Cell 1).")